# latin-mv-tlt — train the v0.2 translation model

Replaces `colab_train_realize.ipynb`. Two structural differences:

- **one** training run, not two. v0.2 uses a single model for both directions,
  selected by a task prefix (R-3.1).
- an **export cell**. v0.1 had none, so the ONNX conversion happened outside any
  committed tooling — which is how 307 MB of models ended up containing ~162 MB
  of graphs the runtime never loads. The size gate (R-3.4) now runs here, at M-4,
  before anything is integrated.

Before you start, on your laptop (not here — the corpus builder needs Node for
the transliterator, R-2.2):

```
python tools/build_translation_pairs.py          # M-1
python tools/measure_roundtrip.py --n 1000       # M-2   gate: latin-stable >= 98%
python tools/profile_tokenizer.py                # M-2b  gate: <unk> <= 5%
```

Upload `train.jsonl` and `valid.jsonl` when prompted below.


## 1. GPU


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'
print(torch.cuda.get_device_name(0))


## 2. Dependencies

Mirrors `tools/requirements.txt`. `transformers>=4.46` is required for
`processing_class` on the trainer.


In [ ]:
%pip install -q -U 'transformers>=4.46' 'accelerate>=1.1' sentencepiece \
    'sacrebleu>=2.4' 'optimum[onnxruntime]>=1.23' 'onnx>=1.17' 'onnxruntime>=1.19'

# Colab preinstalls a `diffusers` that is incompatible with its `huggingface_hub`
# (`cannot import name 'get_cached_repo_tree'`). optimum imports diffusers behind
# `is_diffusers_available()`, so that breakage takes down the ONNX export of a
# seq2seq model that has nothing to do with diffusion. Nothing here uses it.
%pip uninstall -y -q diffusers


## 3. Get the code and the corpus

The training and export scripts are committed, so they are cloned rather than
pasted — a notebook copy of the trainer is a second implementation to keep in
sync, which is exactly what went wrong in v0.1 (`FrameDataset` existed verbatim
in both the script and the notebook).


In [ ]:
import os
REPO = 'https://github.com/Wildeys/latin-mv-tlt.git'   # adjust if your remote differs
if not os.path.exists('latin-mv-tlt'):
    !git clone --depth 1 $REPO
%cd latin-mv-tlt
!mkdir -p data/parallel


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload train.jsonl and valid_small.jsonl to Drive from your browser once.
# files.upload() stalls or dies at 189 MB, and a dropped session would make you
# do it again. valid_small.jsonl is the 2,000-row subset from TRAINING.md step 3
# — generating over all 49,948 valid rows costs 30-50 min per evaluation.
!cp /content/drive/MyDrive/train.jsonl /content/drive/MyDrive/valid_small.jsonl data/parallel/
!ls -la data/parallel/


## 4. Verify the corpus before training

v0.1's notebook asserted hard-coded line counts, which broke every time the
corpus legitimately changed. These checks assert *properties* instead: the
prefix is present, both directions are represented, and no Thaana reached the
model input — that last one is the architecture's whole premise (§1.1).


In [ ]:
import json, re, collections

def load(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

train = load('data/parallel/train.jsonl')
valid = load('data/parallel/valid_small.jsonl')
THAANA = re.compile('[\u0780-\u07BF]')

for name, rows in (('train', train), ('valid', valid)):
    assert rows, f'{name} is empty'
    dirs = collections.Counter(r['direction'] for r in rows)
    assert set(dirs) == {'dv-en', 'en-dv'}, f'{name}: expected both directions, got {dict(dirs)}'
    for r in rows:
        assert r['input'].startswith('translate '), f'unprefixed row: {r["input"][:60]}'
        assert not THAANA.search(r['input']), 'Thaana reached the model input'
    print(f'{name:<6} {len(rows):>7} rows  {dict(dirs)}')

train_inputs = {r['input'] for r in train}
leaked = sum(1 for r in valid if r['input'] in train_inputs)
assert leaked == 0, f'{leaked} valid inputs also appear in train (R-2.6)'
print('no train/valid leakage')


## 5. Train (M-3)

`tools/train_translate.py` carries the hyperparameters from R-9.2: **lr 1e-4**
(not v0.1's 5e-5, and not the 3e-4 the spec first suggested), weight decay 0.01,
batch 32, and `load_best_model_at_end` on chrF++ — without which `save_model()`
keeps the *last* epoch rather than the best.

`SAVE_STEPS` and `RESUME` are what make a dropped session survivable. Saving
once per epoch is ~15,000 steps at batch 32, so a timeout at step 50,000 throws
away everything since 45,003; `SAVE_STEPS = 5000` bounds that. `RESUME = 'auto'`
takes the newest checkpoint in `<OUT>/runs`, and **fails** if there is none
rather than quietly spending four hours training from scratch because Drive was
not mounted. Set it to `None` when you mean to start over.

Set `SMOKE = True` for a 30-second wiring check. R-8.3: a smoke checkpoint must
never be demoed or evaluated as a result.


### Checkpoints live in Drive

`OUT` is a Drive path, not a directory in the clone. Colab's own disk dies with
the session, so a checkpoint written under `latin-mv-tlt/` is gone the moment the
tab drops — and at ~730 MB each that is hours of GPU time.

Resuming needs **every** checkpoint of the interrupted run in `<OUT>/runs`, not
just the newest. `trainer_state.json` records the best checkpoint as an absolute
path from the session that wrote it; transformers rebuilds that path under
`output_dir` on each save, and when the directory is missing it only *warns* —
then `save_model()` ships the last step instead of the best one.
`train_translate.py` refuses to start rather than let that happen quietly.

Run this to see what the trainer can reach, and `!mv` anything stranded
elsewhere in Drive into `runs/`.


In [ ]:
import glob, os
OUT = '/content/drive/MyDrive/dv-en-translate'
RUNS = f'{OUT}/runs'
os.makedirs(RUNS, exist_ok=True)

print('the trainer can resume from:')
for d in sorted(glob.glob(f'{RUNS}/checkpoint-*')):
    print('  ', os.path.basename(d))

stranded = {d for pat in ('/content/drive/MyDrive/checkpoint-*',
                          '/content/drive/MyDrive/*/checkpoint-*',
                          '/content/drive/MyDrive/*/*/checkpoint-*')
            for d in glob.glob(pat) if not d.startswith(RUNS)}
if stranded:
    print('\nelsewhere in Drive — move these in if they belong to this run:')
    for d in sorted(stranded):
        print('  ', d)
    print(f"\n  !mv '<path>' '{RUNS}/'")


In [ ]:
SMOKE = False
BATCH = 32          # drop to 8-16 on CUDA OOM, or for flan-t5-base
SAVE_STEPS = 5000   # ~15,000 steps per epoch, so an interruption costs <= 5,000
RESUME = 'auto'     # None for a fresh run; 'auto' after a dropped session

!python tools/train_translate.py \
    --train data/parallel/train.jsonl \
    --valid data/parallel/valid_small.jsonl \
    --out {OUT} \
    --model t5-small --epochs 4 --batch {BATCH} --lr 1e-4 \
    --save-steps {SAVE_STEPS} \
    {'--resume ' + RESUME if RESUME else ''} \
    {'--smoke' if SMOKE else ''}


### Per-epoch metrics

R-8.5. Record this table for the write-up: it is the evidence that the best
checkpoint was selected on a metric rather than on the last epoch.


In [ ]:
OUT = '/content/drive/MyDrive/dv-en-translate'
import json
stats = json.load(open(f'{OUT}/training_stats.json'))
for row in stats['perEpoch']:
    print({k: v for k, v in row.items() if k.startswith('eval_') or k == 'epoch'})
print('\nbest chrF++:', stats['bestMetric'])


## 6. Probe


In [ ]:
OUT = '/content/drive/MyDrive/dv-en-translate'
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(OUT)
mdl = AutoModelForSeq2SeqLM.from_pretrained(OUT).cuda().eval()

def go(text):
    ids = tok(text, return_tensors='pt').input_ids.cuda()
    out = mdl.generate(ids, max_new_tokens=128, num_beams=1, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

print(go('translate Dhivehi Latin to English: aharen maleah dhaanan'))
print(go('translate English to Dhivehi Latin: I will go to Male.'))


## 6b. Trim the vocabulary (R-3.2 step 1)

The honest export lands at **80.27 MB** against an 80.00 MB budget — which
`docs/DESIGN.md` §5.3 predicted ("35 + 42 + 2.4 ≈ 80 MB — *at* the budget, not
under it"). `lm_head` is already deduplicated against `shared.weight`, so the
only lever left is the embedding: 32,128 × 512 appears once per graph, about 42%
of the shipped ONNX.

Measured over all 1,141,116 corpus texts, t5's vocabulary is **73.2% used** —
23,404 distinct ids. Dropping the other 8,623 rows frees ~8.8 MB across the two
graphs and brings the export to ~71 MB.

**This does not require retraining.** Slicing keeps each surviving token's
learned vector bit for bit; the model only loses the ability to emit what was
dropped. `trim_vocab.py` proves that rather than assuming it — it re-tokenizes
sampled corpus rows and refuses to write if a single piece sequence changed, then
generates from both models and refuses to write if any output differs.

Pass **all three splits**. A token that only appears in `test.jsonl` would
otherwise be dropped, breaking M-10 evaluation later. They are small (18 MB and
15 MB) — upload them to Drive alongside `train.jsonl`.


In [ ]:
!cp /content/drive/MyDrive/valid.jsonl /content/drive/MyDrive/test.jsonl data/parallel/ 2>/dev/null
!ls -la data/parallel/*.jsonl

TRIMMED = OUT + '-trimmed'
!python tools/trim_vocab.py \
    --model {OUT} \
    --corpus data/parallel/train.jsonl data/parallel/valid.jsonl data/parallel/test.jsonl \
    --out {TRIMMED}


## 7. Export to ONNX INT8 (M-4)

This is the cell v0.1 never had. `tools/export_onnx.py` asserts at every step:

- the merged decoder really exposes `use_cache_branch` — the assertion that makes
  deleting the old `runBeam` monkey-patch safe;
- each quantized graph actually shrank, which catches `quantize_dynamic` silently
  skipping `If` subgraphs and leaving the file fp32 inside;
- the total is within the 80 MB budget, and it **refuses to write** if not.

If the budget gate fails, the contingency ladder is in the error message and in
REQUIREMENTS.md R-3.2. Vocabulary trimming is the big win and has to happen
*before* retraining, so do not skip past this.


In [ ]:
import sys

TRIMMED = '/content/drive/MyDrive/dv-en-translate-trimmed'
# `!{sys.executable}`, not `!python`: %pip installed optimum into the kernel's
# interpreter, and a bare `python` may not be that interpreter.
!{sys.executable} tools/export_onnx.py \
    --model {TRIMMED} \
    --out public/models/dv-en-translate


In [ ]:
import json
print(json.dumps(json.load(open('public/models/dv-en-translate/export_stats.json')), indent=2))


## 8. Download

Unzip into `public/models/dv-en-translate/` in your working tree, then:

```
node tools/smoke_translate.mjs 'aharen maleah dhaanan'   # runs the export under Node
npm run check:models                                     # the same budget gate, in CI
npm run dev                                              # the only check that exercises WASM
```

Only after that is the v0.2 model verified, and only then does M-8b delete the
v0.1 realization models.


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('dv-en-translate', 'zip', 'public/models/dv-en-translate')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy('dv-en-translate.zip', '/content/drive/MyDrive/')
    print('copied to Drive')
except Exception as exc:
    print('Drive copy skipped:', exc)
files.download('dv-en-translate.zip')
